# Weather Station Data: Merge, Visualize, Post-Process

Built for NC State Climate Office (ECONET) 1-minute exports with columns:
`Date/Time (Eastern)`, `Air Temperature (C)`, `Relative Humidity (%)`, `Solar Radiation (W/m2)`.

**How to use:** put all station files (`.xlsx` or `.csv`) in one folder, set `DATA_DIR` below, and run all cells.

**Expected folder layout** (run the notebooks from `senior_design/`):
```
senior_design/
├── summer_data/        1-minute station files (.xlsx)
├── rain_data/          hourly precipitation file (.xlsx)
├── output/             created by weather_station_analysis.ipynb
│   ├── combined_weather_data.csv
│   ├── tuning/         solar_temp_tuning.ipynb results
│   └── sun_warming/    sun_warming_estimate.ipynb results
├── weather_station_analysis.ipynb
├── solar_temp_tuning.ipynb
└── sun_warming_estimate.ipynb
```

Sections:
1. Settings
2. Load original datasets (reads first and last timestamp of each file) and the hourly rain file
3. Concatenate into one continuous time series
4. Data visualization (one temperature and humidity line graph per month, rainy hours shaded light green)
5. Post-processing (daylight window, average temperatures, solar radiation vs temperature rise)
6. Export: every graph is saved as a PNG and every section 5 table goes into one Excel workbook, all inside the `output` folder

## 1. Settings

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats

# ---- edit these ----
DATA_DIR        = "./summer_data" # folder holding every 1-minute station file
FILE_PATTERNS   = ["*.xlsx", "*.xls", "*.csv"]
SOLAR_THRESHOLD = 0.0             # W/m2; solar above this counts as "measurable"
RISE_INTERVAL   = "30min"         # window used to measure how fast temperature rises

# hourly precipitation file (if a copy sits in DATA_DIR too, the loader skips it automatically)
RAIN_DIR        = "./rain_data"   # folder with the hourly precipitation file
RAIN_FILE       = sorted(glob.glob(os.path.join(RAIN_DIR, "*.xls*")) + glob.glob(os.path.join(RAIN_DIR, "*.csv")))[0]
RAIN_THRESHOLD  = 0.0             # cm; an hour with more rain than this is shaded as rainy
RAIN_LABEL      = "start"         # "start": 14:00 row = rain from 14:00-15:00; "end": rain from 13:00-14:00
RAIN_COLOR      = "#b8e6a0"       # light green overlay
OUTPUT_DIR      = "output"        # every saved file (graphs, tables, merged data) goes here
FIG_DPI         = 200             # resolution of saved PNG graphs

# standard column names used through the notebook
TIME, TEMP, RH, SOLAR = "time", "temp_C", "rh_pct", "solar_Wm2"

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.grid": True, "grid.alpha": 0.3})
TEMP_COLOR, RH_COLOR, SOLAR_COLOR = "#c8553d", "#2f6690", "#e0a100"

os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV  = os.path.join(OUTPUT_DIR, "combined_weather_data.csv")
OUTPUT_XLSX = os.path.join(OUTPUT_DIR, "post_processing_results.xlsx")

saved_figures = []          # list of PNG paths written by save_fig
export_tables = {}          # sheet name -> (description, DataFrame) for the Excel export

def save_fig(fig, name):
    """Save a figure as a PNG in OUTPUT_DIR and remember the path."""
    path = os.path.join(OUTPUT_DIR, f"{name}.png")
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    saved_figures.append(path)
    return path

def add_table(sheet, description, table):
    """Queue a table for the Excel workbook (sheet names max 31 characters)."""
    export_tables[sheet[:31]] = (description, table)


## 2. Load original datasets

Each file has a block of station metadata above the data. The loader finds the header row (the one starting with `Date/Time`), renames the columns, and turns QC flags like `QCF` into blank values (NaN) so they don't break the math. It then records the first and last timestamp in each file.

In [ ]:
def find_header_row(raw):
    """Return the index of the row whose first cell starts with 'Date'."""
    for i, v in raw.iloc[:, 0].items():
        if isinstance(v, str) and v.strip().lower().startswith("date"):
            return i
    raise ValueError("Could not find a 'Date/Time' header row")


def load_station_file(path):
    ext = os.path.splitext(path)[1].lower()
    reader = pd.read_csv if ext == ".csv" else pd.read_excel
    raw = reader(path, header=None)
    hdr = find_header_row(raw)
    df = raw.iloc[hdr + 1:].copy()
    df.columns = [str(c).strip() for c in raw.iloc[hdr]]

    # map the original column names to short names
    rename = {}
    for c in df.columns:
        lc = c.lower()
        if lc.startswith("date"):        rename[c] = TIME
        elif "temperature" in lc:        rename[c] = TEMP
        elif "humidity" in lc:           rename[c] = RH
        elif "solar" in lc:              rename[c] = SOLAR
    df = df.rename(columns=rename)
    if not {TIME, TEMP, RH, SOLAR}.issubset(df.columns):
        return None                                  # not a temp/RH/solar file
    df = df[[TIME, TEMP, RH, SOLAR]]

    df[TIME] = pd.to_datetime(df[TIME], errors="coerce")
    for c in (TEMP, RH, SOLAR):
        df[c] = pd.to_numeric(df[c], errors="coerce")   # 'QCF' etc. -> NaN
    df = df.dropna(subset=[TIME]).sort_values(TIME).reset_index(drop=True)
    df["source_file"] = os.path.basename(path)
    return df


files = sorted({f for p in FILE_PATTERNS for f in glob.glob(os.path.join(DATA_DIR, p))})
if not files:
    raise FileNotFoundError(f"No data files found in {os.path.abspath(DATA_DIR)}")

datasets, summary = [], []
for f in files:
    d = load_station_file(f)
    if d is None:
        print(f"Skipped {os.path.basename(f)}: no temperature/humidity/solar columns")
        continue
    datasets.append(d)
    summary.append({
        "file": os.path.basename(f),
        "first_time": d[TIME].iloc[0],
        "last_time": d[TIME].iloc[-1],
        "rows": len(d),
        "flagged_temp": int(d[TEMP].isna().sum()),
        "flagged_rh": int(d[RH].isna().sum()),
        "flagged_solar": int(d[SOLAR].isna().sum()),
    })

file_summary = pd.DataFrame(summary).sort_values("first_time").reset_index(drop=True)
print(f"Loaded {len(datasets)} file(s)")
file_summary

### 2b. Load hourly rain data

Reads the hourly `Total Precipitation (cm)` file. Every hour with more than `RAIN_THRESHOLD` cm counts as a rainy hour. Back-to-back rainy hours are joined into one **rain event** so the graphs can shade each event as one block.

In [ ]:
def load_rain_file(path):
    ext = os.path.splitext(path)[1].lower()
    raw = (pd.read_csv if ext == ".csv" else pd.read_excel)(path, header=None)
    hdr = find_header_row(raw)
    r = raw.iloc[hdr + 1:, :2].copy()
    r.columns = [TIME, "rain_cm"]
    r[TIME] = pd.to_datetime(r[TIME], errors="coerce")
    r["rain_cm"] = pd.to_numeric(r["rain_cm"], errors="coerce")
    return r.dropna(subset=[TIME]).set_index(TIME).sort_index()

rain = load_rain_file(RAIN_FILE)
hour = pd.Timedelta(hours=1)
offset = pd.Timedelta(0) if RAIN_LABEL == "start" else -hour
rain["hour_start"] = rain.index + offset
rain["hour_end"] = rain["hour_start"] + hour
rain["is_rain"] = rain["rain_cm"] > RAIN_THRESHOLD

# join consecutive rainy hours into events
wet = rain[rain["is_rain"]]
event_id = (wet["hour_start"] != wet["hour_end"].shift()).cumsum()
rain_events = (wet.groupby(event_id)
                  .agg(start=("hour_start", "first"), end=("hour_end", "last"),
                       rainy_hours=("rain_cm", "size"), total_rain_cm=("rain_cm", "sum"),
                       peak_hour_cm=("rain_cm", "max"))
                  .reset_index(drop=True))
rain_events.index = rain_events.index + 1
rain_events.index.name = "event"

print(f"Rain file: {rain.index.min()} to {rain.index.max()}  ({len(rain):,} hours)")
print(f"Rainy hours: {int(rain['is_rain'].sum())}   rain events: {len(rain_events)}   "
      f"total rain: {rain['rain_cm'].sum():.2f} cm")
rain_events.head(10)

## 3. Concatenate into one continuous time series

Files are ordered by their first timestamp. For each pair of neighbouring files the cell checks whether the next file starts right after the previous one ends, and reports any **gap** or **overlap**. Duplicate timestamps from overlaps are dropped (first file wins). The result is put on a regular time grid so any missing minutes show up as blank rows instead of being silently skipped.

In [ ]:
# order datasets the same way as the summary table
order = {name: i for i, name in enumerate(file_summary["file"])}
datasets.sort(key=lambda d: order[d["source_file"].iloc[0]])

# sampling interval (most common step inside the files, e.g. 1 minute)
step = pd.concat([d[TIME].diff() for d in datasets]).mode().iloc[0]
print("Sampling interval:", step)

# check how each file joins onto the previous one
joins = []
for prev, nxt in zip(file_summary.itertuples(), file_summary.iloc[1:].itertuples()):
    gap = nxt.first_time - prev.last_time
    status = "continuous" if gap == step else ("OVERLAP" if gap < step else "GAP")
    joins.append({"from": prev.file, "to": nxt.file,
                  "prev_end": prev.last_time, "next_start": nxt.first_time,
                  "difference": gap, "status": status})
join_report = pd.DataFrame(joins)
if len(join_report):
    display(join_report)

combined = (pd.concat(datasets, ignore_index=True)
              .sort_values(TIME)
              .drop_duplicates(subset=TIME, keep="first")
              .set_index(TIME))

# regular grid from first to last timestamp
full_index = pd.date_range(combined.index.min(), combined.index.max(), freq=step)
missing = full_index.difference(combined.index)
combined = combined.reindex(full_index)
combined.index.name = TIME

print(f"Combined: {combined.index.min()}  to  {combined.index.max()}")
print(f"Rows: {len(combined):,}   missing timestamps filled as NaN: {len(missing):,}")
combined.to_csv(OUTPUT_CSV)
print("Saved", OUTPUT_CSV)
combined.head()

## 4. Data visualization

One line graph per month: temperature (left axis) and relative humidity (right axis) against time. Hours with rain are shaded light green. Gaps in the data show up as breaks in the lines. Each graph is saved to the `output` folder as `temp_rh_<YYYY-MM>.png`.

In [ ]:
from matplotlib.patches import Patch

def shade_rain(ax, t0, t1):
    """Shade every rain event that overlaps the window t0 to t1. Returns the number shaded."""
    ev = rain_events[(rain_events["end"] > t0) & (rain_events["start"] < t1)]
    for e in ev.itertuples():
        ax.axvspan(max(e.start, t0), min(e.end, t1), color=RAIN_COLOR, alpha=0.6, lw=0, zorder=0)
    return len(ev)

RAIN_PATCH = Patch(facecolor=RAIN_COLOR, alpha=0.6, label="Rain (hourly)")

def dual_line(df, title):
    fig, ax1 = plt.subplots(figsize=(13, 5))
    ax2 = ax1.twinx()
    ax1.plot(df.index, df[TEMP], lw=0.9, color=TEMP_COLOR, label="Air temperature (°C)")
    ax2.plot(df.index, df[RH],   lw=0.9, color=RH_COLOR,   label="Relative humidity (%)")
    ax1.set_ylabel("Air temperature (°C)", color=TEMP_COLOR)
    ax2.set_ylabel("Relative humidity (%)", color=RH_COLOR)
    ax1.tick_params(axis="y", colors=TEMP_COLOR)
    ax2.tick_params(axis="y", colors=RH_COLOR)
    ax2.grid(False); ax2.spines["top"].set_visible(False)
    ax1.set_xlabel("Time (Eastern)")
    t0, t1 = df.index.min(), df.index.max()
    shade_rain(ax1, t0, t1)
    ax1.set_xlim(t0, t1)
    loc = mdates.AutoDateLocator()
    ax1.xaxis.set_major_locator(loc)
    ax1.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
    h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2 + [RAIN_PATCH], l1 + l2 + ["Rain (hourly)"], loc="upper center",
               bbox_to_anchor=(0.5, -0.14), ncol=3, frameon=False)
    ax1.set_title(title)
    fig.tight_layout()
    return fig

for month, g in combined.groupby(combined.index.to_period("M")):
    if g[[TEMP, RH]].dropna(how="all").empty:
        continue                                   # skip months with no data at all
    fig = dual_line(g, f"Temperature and relative humidity, {month.strftime('%B %Y')}")
    save_fig(fig, f"temp_rh_{month}")
    plt.show()

## 5. Post-processing

### 5a. Daylight window from solar radiation
For each day, the first and last timestamp where solar radiation is above `SOLAR_THRESHOLD` mark the start and end of measurable sunlight.

In [ ]:
df = combined.copy()
df["date"] = df.index.normalize()
df["month"] = df.index.to_period("M")

sunny = df[df[SOLAR] > SOLAR_THRESHOLD]
daylight = (sunny.reset_index()
                 .groupby("date")[TIME]
                 .agg(first_solar="min", last_solar="max"))
daylight["daylight_hours"] = (daylight["last_solar"] - daylight["first_solar"]).dt.total_seconds() / 3600

# flag each row as inside or outside that day's daylight window
df = df.join(daylight[["first_solar", "last_solar"]], on="date")
df["is_daylight"] = (df.index >= df["first_solar"]) & (df.index <= df["last_solar"])

add_table("Daylight window",
          f"First and last time each day with solar radiation above {SOLAR_THRESHOLD} W/m2, "
          "and the hours between them.",
          daylight)
daylight

### 5b. Average temperatures
Averages are given for all hours and for daylight hours only, at three scales:
- **Whole time scale:** every data point in the combined dataset
- **Monthly:** grouped by calendar month
- **Peaks:** the daily maximum temperature, averaged (overall and per month)

In [ ]:
# daily peaks (max temperature each day and when it happened)
valid = df.dropna(subset=[TEMP])            # skip days that are fully missing
daily = valid.groupby("date").agg(peak_temp_C=(TEMP, "max"),
                               min_temp_C=(TEMP, "min"),
                               mean_temp_C=(TEMP, "mean"),
                               peak_solar_Wm2=(SOLAR, "max"))
daily["peak_time"] = valid.groupby("date")[TEMP].idxmax()
daily["month"] = daily.index.to_period("M")

day = df[df["is_daylight"]]

overall = pd.DataFrame({
    "all hours (°C)":     [df[TEMP].mean()],
    "daylight only (°C)": [day[TEMP].mean()],
    "mean daily peak (°C)": [daily["peak_temp_C"].mean()],
    "highest peak (°C)":  [daily["peak_temp_C"].max()],
    "highest peak at":    [daily.loc[daily["peak_temp_C"].idxmax(), "peak_time"]],
}, index=pd.Index(["whole time scale"], name="period"))

monthly = pd.DataFrame({
    "all hours (°C)":       df.groupby("month")[TEMP].mean(),
    "daylight only (°C)":   day.groupby("month")[TEMP].mean(),
    "mean daily peak (°C)": daily.groupby("month")["peak_temp_C"].mean(),
    "highest peak (°C)":    daily.groupby("month")["peak_temp_C"].max(),
})

r2 = lambda d: d.apply(lambda c: c.round(2) if c.dtype.kind == "f" else c)
print("Whole time scale"); display(r2(overall))
print("By month");         display(r2(monthly))
print("Daily summary");    display(r2(daily))

add_table("Avg temp - whole period",
          "Average air temperature over the full combined dataset: all hours, daylight hours only, "
          "and the average of each day's peak (maximum) temperature.", overall)
add_table("Avg temp - monthly",
          "Average air temperature for each month: all hours, daylight hours only, mean daily peak, "
          "and the highest peak of the month.", monthly)
add_table("Daily summary",
          "Per-day peak, minimum and mean temperature, peak solar radiation, and the time of the "
          "temperature peak.", daily)

### 5c. Solar radiation vs temperature rise (linear correlation)

The data is split into `RISE_INTERVAL` blocks during daylight. For each block:
- **x** = mean solar radiation (W/m²)
- **y** = rate of temperature change (°C per hour) = (last temp − first temp) / block length

A straight line `rise = slope × solar + intercept` is fitted with least squares. If there are three or more days, a second fit compares each day's total solar energy to how much the temperature rose from sunrise to the daily peak.

In [ ]:
def interval_rise(g):
    t = g[TEMP].dropna()
    if len(t) < 2:
        return pd.Series({"solar_Wm2": np.nan, "rise_C_per_hr": np.nan})
    hours = (t.index[-1] - t.index[0]).total_seconds() / 3600
    return pd.Series({"solar_Wm2": g[SOLAR].mean(),
                      "rise_C_per_hr": (t.iloc[-1] - t.iloc[0]) / hours})

blocks = (day.groupby(pd.Grouper(freq=RISE_INTERVAL))
             .apply(interval_rise)
             .dropna())

fit = stats.linregress(blocks["solar_Wm2"], blocks["rise_C_per_hr"])
print(f"Temperature rise (°C/hr) = {fit.slope:.5f} × Solar (W/m²) + {fit.intercept:.3f}")
print(f"r = {fit.rvalue:.3f}   R² = {fit.rvalue**2:.3f}   p = {fit.pvalue:.2e}   n = {len(blocks)} blocks")
print(f"Each extra 100 W/m² of sun adds about {fit.slope*100:.2f} °C/hr to the warming rate")

def rise_from_solar(solar_Wm2):
    """Predicted temperature rise rate (°C/hr) for a given solar radiation."""
    return fit.slope * np.asarray(solar_Wm2) + fit.intercept

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(blocks["solar_Wm2"], blocks["rise_C_per_hr"], s=22, color=SOLAR_COLOR,
           edgecolor="none", alpha=0.8, label=f"{RISE_INTERVAL} daylight blocks")
xs = np.linspace(0, blocks["solar_Wm2"].max(), 100)
ax.plot(xs, rise_from_solar(xs), color="#333333", lw=2,
        label=f"fit: y = {fit.slope:.4f}x + {fit.intercept:.2f}  (R² = {fit.rvalue**2:.2f})")
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("Mean solar radiation (W/m²)")
ax.set_ylabel("Temperature change (°C per hour)")
ax.set_title("Solar radiation vs rate of temperature rise")
ax.legend(frameon=False)
fig.tight_layout()
save_fig(fig, "solar_vs_temp_rise_rate")
plt.show()

fit_results = [{"fit": f"{RISE_INTERVAL} daylight blocks",
                "x": "mean solar radiation (W/m2)", "y": "temperature change (C/hr)",
                "slope": fit.slope, "intercept": fit.intercept, "r": fit.rvalue,
                "R2": fit.rvalue**2, "p_value": fit.pvalue, "n": len(blocks)}]
add_table("Rise rate blocks",
          f"Each row is one {RISE_INTERVAL} daylight block: mean solar radiation and how fast the "
          "temperature changed (C per hour). These points feed the linear fit.", blocks)

In [ ]:
# Daily version: total solar energy vs temperature rise from sunrise to peak
rows = []
for d, g in valid.groupby("date"):
    if d not in daylight.index:
        continue
    start = daylight.loc[d, "first_solar"]
    t_start = g[TEMP].loc[start:].dropna()
    if t_start.empty:
        continue
    rows.append({"date": d,
                 "insolation_MJ_m2": g[SOLAR].fillna(0).sum() * step.total_seconds() / 1e6,
                 "temp_at_first_solar_C": t_start.iloc[0],
                 "peak_temp_C": g[TEMP].max(),
                 "temp_rise_C": g[TEMP].max() - t_start.iloc[0]})
daily_rise = pd.DataFrame(rows).set_index("date")
display(daily_rise.round(2))

if len(daily_rise) >= 3:
    dfit = stats.linregress(daily_rise["insolation_MJ_m2"], daily_rise["temp_rise_C"])
    print(f"Daily temp rise (°C) = {dfit.slope:.3f} × Insolation (MJ/m²) + {dfit.intercept:.2f}"
          f"   R² = {dfit.rvalue**2:.3f}   p = {dfit.pvalue:.2e}")
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(daily_rise["insolation_MJ_m2"], daily_rise["temp_rise_C"], color=SOLAR_COLOR, s=30)
    xs = np.linspace(daily_rise["insolation_MJ_m2"].min(), daily_rise["insolation_MJ_m2"].max(), 50)
    ax.plot(xs, dfit.slope * xs + dfit.intercept, color="#333333", lw=2,
            label=f"R² = {dfit.rvalue**2:.2f}")
    ax.set_xlabel("Daily solar energy (MJ/m²)")
    ax.set_ylabel("Temperature rise, first sun to peak (°C)")
    ax.set_title("Daily solar energy vs daily temperature rise")
    ax.legend(frameon=False); fig.tight_layout()
    save_fig(fig, "daily_solar_vs_temp_rise")
    plt.show()
    fit_results.append({"fit": "daily totals",
                        "x": "daily solar energy (MJ/m2)", "y": "temp rise, first sun to peak (C)",
                        "slope": dfit.slope, "intercept": dfit.intercept, "r": dfit.rvalue,
                        "R2": dfit.rvalue**2, "p_value": dfit.pvalue, "n": len(daily_rise)})
else:
    print("Daily fit skipped: needs at least 3 days of data.")

add_table("Rain events",
          f"Each row is one rain event: back-to-back hours with more than {RAIN_THRESHOLD} cm of rain. "
          "Start and end bound the shaded green blocks on the graphs.", rain_events)
monthly_rain = rain.groupby(rain["hour_start"].dt.to_period("M")).agg(
    total_rain_cm=("rain_cm", "sum"), rainy_hours=("is_rain", "sum"))
monthly_rain.index.name = "month"
monthly_rain["rain_events"] = rain_events.groupby(rain_events["start"].dt.to_period("M")).size()
monthly_rain = monthly_rain.fillna({"rain_events": 0}).astype({"rain_events": int})
add_table("Rain by month",
          "Total rain (cm), number of rainy hours and number of rain events in each month.",
          monthly_rain)
add_table("Daily temp rise",
          "Per-day total solar energy (MJ/m2), temperature at first measurable sun, peak temperature, "
          "and the rise between them.", daily_rise)
add_table("Linear fit results",
          "Linear regression results (y = slope * x + intercept) for solar radiation vs temperature "
          "rise. r is the correlation coefficient, R2 the share of variation explained.",
          pd.DataFrame(fit_results).set_index("fit"))

### 5d. Optional: one-day check
Plots temperature and solar radiation for a single day with the daylight window shaded grey and rainy hours shaded light green. Change `CHECK_DAY` to look at any date.

In [ ]:
CHECK_DAY = daylight.index[0]          # e.g. pd.Timestamp("2026-07-28")
g = df[df["date"] == CHECK_DAY]

fig, ax1 = plt.subplots(figsize=(12, 4.5))
ax2 = ax1.twinx()
ax1.scatter(g.index, g[TEMP], s=3, color=TEMP_COLOR, label="Air temperature (°C)")
ax2.fill_between(g.index, g[SOLAR], color=SOLAR_COLOR, alpha=0.25, label="Solar radiation (W/m²)")
ax1.axvspan(daylight.loc[CHECK_DAY, "first_solar"], daylight.loc[CHECK_DAY, "last_solar"],
            color="grey", alpha=0.08)
shade_rain(ax1, CHECK_DAY, CHECK_DAY + pd.Timedelta(days=1))
pk = daily.loc[CHECK_DAY]
ax1.margins(y=0.12)
ax1.annotate(f"peak {pk.peak_temp_C:.1f} °C", (pk.peak_time, pk.peak_temp_C),
             xytext=(10, 8), textcoords="offset points")
ax1.set_ylabel("Air temperature (°C)", color=TEMP_COLOR)
ax2.set_ylabel("Solar radiation (W/m²)", color="#8a6400")
ax2.grid(False)
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2 + [RAIN_PATCH], l1 + l2 + ["Rain (hourly)"], loc="upper center",
           bbox_to_anchor=(0.5, -0.14), ncol=3, markerscale=4, frameon=False)
ax1.set_title(f"{CHECK_DAY:%Y-%m-%d}: daylight window "
              f"{daylight.loc[CHECK_DAY,'first_solar']:%H:%M} to {daylight.loc[CHECK_DAY,'last_solar']:%H:%M}")
fig.tight_layout()
save_fig(fig, f"day_check_{CHECK_DAY:%Y-%m-%d}")
plt.show()

## 6. Export to the `output` folder

Writes every section 5 table to one Excel workbook. The first tab, **Contents**, lists each tab with its description; each table tab also has its description in the top row. Graphs were already saved as PNGs by the cells above.

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

FONT       = Font(name="Arial", size=10)
BOLD       = Font(name="Arial", size=10, bold=True)
TITLE      = Font(name="Arial", size=12, bold=True)
DESC       = Font(name="Arial", size=10, italic=True, color="555555")
HEAD_FILL  = PatternFill("solid", fgColor="DCE6F1")

def excel_ready(table):
    """Make index/columns plain values Excel can store (periods and timedeltas as text)."""
    t = table.reset_index()
    for c in t.columns:
        if isinstance(t[c].dtype, pd.PeriodDtype) or t[c].dtype.kind == "m":
            t[c] = t[c].astype(str)
        elif t[c].dtype.kind == "f" and c != "p_value":
            t[c] = t[c].round(6)
    return t

def style_sheet(ws, title, description, n_cols, header_row):
    ws["A1"] = title;       ws["A1"].font = TITLE
    ws["A2"] = description; ws["A2"].font = DESC
    ws["A2"].alignment = Alignment(wrap_text=True, vertical="top")
    ws.merge_cells(start_row=2, start_column=1, end_row=2, end_column=max(n_cols, 6))
    ws.row_dimensions[2].height = 32
    headers = [c.value for c in ws[header_row]]
    for row in ws.iter_rows(min_row=header_row + 1):
        for cell in row:
            cell.font = FONT
            name = headers[cell.column - 1] if cell.column <= len(headers) else None
            if hasattr(cell.value, "hour"):
                cell.number_format = "yyyy-mm-dd" if name == "date" else "yyyy-mm-dd hh:mm"
            elif name == "p_value":
                cell.number_format = "0.00E+00"
    for cell in ws[header_row]:
        cell.font = BOLD; cell.fill = HEAD_FILL
    for i in range(1, n_cols + 1):
        col = get_column_letter(i)
        width = max(len(str(c.value)) for c in ws[col][header_row - 1:] if c.value is not None)
        ws.column_dimensions[col].width = min(max(width + 2, 12), 40)
    ws.freeze_panes = ws.cell(row=header_row + 1, column=1)

HEADER_ROW = 4
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as xw:
    contents = pd.DataFrame([{"tab": k, "description": d, "rows": len(t)}
                             for k, (d, t) in export_tables.items()])
    contents.to_excel(xw, sheet_name="Contents", index=False, startrow=HEADER_ROW - 1)
    for sheet, (desc, table) in export_tables.items():
        excel_ready(table).to_excel(xw, sheet_name=sheet, index=False, startrow=HEADER_ROW - 1)

    ws = xw.sheets["Contents"]
    style_sheet(ws, "Weather station post-processing results",
                f"Source folder: {os.path.abspath(DATA_DIR)}  |  "
                f"Data from {combined.index.min():%Y-%m-%d} to {combined.index.max():%Y-%m-%d}",
                3, HEADER_ROW)
    ws.column_dimensions["B"].width = 90
    for c in ws["B"][HEADER_ROW:]:
        c.alignment = Alignment(wrap_text=True, vertical="top")
    for sheet, (desc, table) in export_tables.items():
        style_sheet(xw.sheets[sheet], sheet, desc, len(excel_ready(table).columns), HEADER_ROW)

print("Saved to", os.path.abspath(OUTPUT_DIR))
for f in [OUTPUT_CSV, OUTPUT_XLSX] + saved_figures:
    print("  ", os.path.basename(f))